# Spike 0.6d — Tier-1 quantitative-sanity counter-checks on Colab

**Spec reference:** `docs/report-final.md §Phase 0 Spike 0.6d` (2026-05-14 addition; H10 supplement).

**Protocol:** `docs/spike_0_6d_protocol.md`.

**Motivation:** `docs/phase_logs/phase_0_signoff.md` Note 2; `docs/phase_logs/spike_0_6c.md` Note 1.

**What this notebook does:**

1. **Sub-spike 0.6d.1 (gating)** — symmetry + dimensional-force sanity on the existing Spike 0.6c Cell 8 history.csv (already on Drive) PLUS one fresh production-Tier-1 run on the NACA 0012 mesh from Spike 0.6c Cell 6. Pass criterion: cycle-averaged force near zero (symmetry) AND dimensional cycle-peak within ±1 order of magnitude of `m × ω² × r_cm`.
2. **Sub-spike 0.6d.2 (gating)** — render the 2D thin-plate pitching cfg (`configs/su2/thin_plate_2d_pitching.cfg.j2`) at production Tier-1 numerics, run SU2 on a 2D thin-plate mesh, compare inviscid-phase moment against the closed-form Sedov/Newman added-mass moment (±15%).
3. **Sub-spike 0.6d.3 (advisory)** — duplicate the production cfg with `SOLVER = INC_NAVIER_STOKES`, run on the same mesh, compare dimensional cycle-averaged forces (±20%).
4. **Aggregator** — `scripts/run_spike_0_6d.py` reads the three result JSONs and writes `data/spike_0_6d/PASS` iff 0.6d.1 AND 0.6d.2 both passed. 0.6d.3 advisory.
5. **Phase 4 gate check** — `scripts/launch_phase4.py --check` returns 0 once BOTH `data/spike_0_6c/PASS` AND `data/spike_0_6d/PASS` exist.

**Compute budget:** ~5–9 hours total on Colab Pro CPU. Counts under Phase 0; NOT against the 1000-h Phase 4 stop rule.

**Prerequisites:** Spike 0.6c notebook must have run at least through Cell 6 (mesh generation) and Cell 8 (NACA 0012 SU2 run that produced `history.csv` on Drive).

## Cell 1 — Imports + constants

All imports for the notebook live in this single cell, at the top, unconditionally. Re-run this cell whenever the runtime restarts.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import drive as colab_drive
from google.colab import userdata as colab_userdata

# 0.6d code is on the feature branch — NOT merged to main yet.
GIT_REPO    = 'https://github.com/clingergab/fan-optimization.git'
GIT_BRANCH  = 'spike-0-6d-tier1-quant-sanity'
GIT_USER    = 'clingergab'
GIT_EMAIL   = 'clinger.gab@gmail.com'
REPO_DIR    = Path('/content/fan-optimization')
DRIVE_DIR   = Path('/content/drive/MyDrive/fan-optimization')
SU2_VERSION = '8.0.1'          # NO leading 'v' — the installer prepends it
SU2_DIR     = Path('/content/su2')
SU2_BIN     = SU2_DIR / 'bin' / 'SU2_CFD'

REPO_ROOT = REPO_DIR  # alias used by later cells

print(f'[setup] REPO_DIR = {REPO_DIR}; branch = {GIT_BRANCH}; SU2_VERSION = {SU2_VERSION}')


## Cell 2 — Clone repo + mount Drive

In [ ]:
colab_drive.mount('/content/drive', force_remount=False)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'[setup] Drive mounted; cache dir {DRIVE_DIR}')

if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--quiet', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_BRANCH], check=True)
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'pull', '--quiet', 'origin', GIT_BRANCH],
        check=True,
    )
branch = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'branch', '--show-current'],
    capture_output=True, text=True,
).stdout.strip()
print(f'[setup] repo at {REPO_DIR} on branch {branch!r}')
assert branch == GIT_BRANCH, f'expected {GIT_BRANCH}, got {branch!r} — 0.6d code may be absent'
os.chdir(REPO_DIR)


## Cell 3 — Install Python dependencies + gmsh native libs

Installs the `fanopt` package (editable), `gmsh` (+ the libGLU/X native libs it dlopens at import — required by `scripts/gen_benchmark_meshes.py` for the 0.6d.2 thin-plate mesh), and the scientific stack. Adds `scripts/` + `src/` to `sys.path` so the in-notebook script imports resolve.

In [ ]:
# Native libs gmsh dlopens at import time (libGLU + X libs). Colab CPU
# runtimes don't ship these by default; without them `import gmsh` raises
# OSError: libGLU.so.1: cannot open shared object file. gmsh is needed by
# scripts/gen_benchmark_meshes.py for the 0.6d.2 thin-plate mesh.
!apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 libxft2 libxinerama1

!pip install --quiet -e {REPO_DIR}
!pip install --quiet gmsh numpy scipy pyyaml jinja2 jsonschema pytest matplotlib pandas

# Make scripts/ + src/ importable so the in-notebook `import run_spike_0_6d_*`
# / `import fanopt.*` calls resolve from the repo working tree.
scripts_path = str(REPO_DIR / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('[deps] fanopt + gmsh + native libs installed; scripts/ + src/ importable')


## Cell 4 — Imports from fanopt + verify

Confirms the Spike 0.6d module landed and exposes the MACH lock constant.


In [ ]:
import fanopt  # noqa: F401
from fanopt.cfd.spike_0_6d import (
    MACH_UNSTEADY_LOCK,
    compute_added_mass_moment_closed_form_2d_plate,
    compute_dimensional_envelope,
)
from fanopt.cfd.configs import (
    render_unsteady_cfg,
    render_thin_plate_2d_pitching_cfg,
)

print(f'[deps] fanopt OK; MACH_UNSTEADY_LOCK = {MACH_UNSTEADY_LOCK} (Round-9 HIGH-12 / C12)')
print('[deps] Spike 0.6d: 0.6d.1 + 0.6d.2 gating, 0.6d.3 advisory')
print('[deps] Phase 4 launch requires BOTH data/spike_0_6c/PASS AND data/spike_0_6d/PASS')

## Cell 5 — Install SU2

Tries paths in order:
1. **Drive cache** — if a prior run cached `SU2_CFD` on Drive, copy it back (instant).
2. **Pre-built binary** from the SU2 GitHub release (Linux x86_64, ~1 min).
3. **Source build** — `meson` + `ninja`, ~30-45 min, last resort.

After install the binary is cached to Drive so future sessions are instant. Required for sub-spike 0.6d.2 (the 2D thin-plate SU2 run). 0.6c.1 recovery + 0.6d.1 do NOT need SU2 — they reuse the existing Drive history.csv.

In [ ]:
SU2_CACHE = DRIVE_DIR / 'su2_cache' / SU2_VERSION


def _su2_works() -> bool:
    if not SU2_BIN.exists():
        return False
    r = subprocess.run([str(SU2_BIN), '--help'], capture_output=True, text=True)
    return r.returncode in (0, 1)  # SU2 --help can exit 1


def _restore_exec_bits() -> None:
    """Drive round-trip drops POSIX +x; re-set it on bin/ + *.so."""
    bin_dir = SU2_DIR / 'bin'
    if bin_dir.is_dir():
        for f in bin_dir.iterdir():
            if f.is_file():
                os.chmod(f, 0o755)
    for so in SU2_DIR.rglob('*.so'):
        if so.is_file():
            os.chmod(so, 0o755)


def _try_drive_cache() -> bool:
    if not SU2_CACHE.exists():
        return False
    SU2_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SU2_CACHE, SU2_DIR, dirs_exist_ok=True)
    _restore_exec_bits()
    print(f'[su2] restored from Drive cache: {SU2_CACHE}')
    return _su2_works()


def _try_binary_release() -> bool:
    url = (
        f'https://github.com/su2code/SU2/releases/download/'
        f'v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip'
    )
    archive = Path('/tmp/su2.zip')
    try:
        urllib.request.urlretrieve(url, archive)
    except Exception as e:
        print(f'[su2] binary download failed: {e}')
        return False
    SU2_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(archive), '-d', str(SU2_DIR)], check=False)
    canonical = SU2_DIR / 'bin' / 'SU2_CFD'
    real = None
    for cand in SU2_DIR.rglob('SU2_CFD'):
        if cand.is_symlink() or not cand.is_file():
            continue
        real = cand
        break
    if real is None:
        print('[su2] no SU2_CFD binary in extracted archive')
        return False
    os.chmod(real, 0o755)
    if canonical.exists() and canonical.resolve() == real.resolve():
        return _su2_works()
    (SU2_DIR / 'bin').mkdir(exist_ok=True)
    if canonical.exists() or canonical.is_symlink():
        canonical.unlink()
    canonical.symlink_to(real)
    return _su2_works()


def _try_source_build() -> bool:
    print('[su2] source build (~30-45 min)...')
    subprocess.run(
        ['apt-get', '-qq', 'install', '-y', 'libopenblas-dev', 'libmpich-dev'],
        check=True,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', 'meson', 'ninja'],
        check=True,
    )
    src = Path('/tmp/SU2-src')
    if not src.exists():
        subprocess.run(
            ['git', 'clone', '--depth=1', '--branch', f'v{SU2_VERSION}',
             'https://github.com/su2code/SU2.git', str(src)],
            check=True,
        )
    subprocess.run(
        [sys.executable, './meson.py', 'build',
         f'-Dprefix={SU2_DIR}', '-Denable-autodiff=false'],
        cwd=src, check=True,
    )
    subprocess.run(['ninja', '-C', 'build', 'install'], cwd=src, check=True)
    return _su2_works()


def _cache_to_drive() -> None:
    if SU2_CACHE.exists():
        return
    print(f'[su2] caching to {SU2_CACHE} for future runs...')
    SU2_CACHE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SU2_DIR, SU2_CACHE)


installed = _try_drive_cache() or _try_binary_release() or _try_source_build()
if not installed:
    raise RuntimeError(
        'SU2 install failed via all paths (cache / binary / source). '
        'Check the GitHub asset name + bump SU2_VERSION if needed.'
    )
_cache_to_drive()
os.environ['PATH'] = f"{SU2_DIR / 'bin'}:{os.environ['PATH']}"
print(f'[su2] SU2_CFD on PATH at {SU2_BIN}')
!SU2_CFD --help | head -5


## Cell 5b — Recover Spike 0.6c.1 PASS marker from existing Drive history.csv

Post-2026-05-14 the 0.6c.1 runner accepts `--su2-history-csv` as evidence from any prior successful SU2 run that used the same Tier-1 numerics (MACH=1e-9 + REF_DIMENSIONALIZATION). The May 2026 Cell 8 history.csv on Drive qualifies, so we recover the 0.6c.1 PASS marker without re-running SU2.

If the recovery file isn't present, the notebook falls back to generating a probe mesh and running SU2 fresh — but that requires SU2_CFD to be installed (Cell 5).

In [ ]:
# Spike 0.6c.1 gate recovery — write data/spike_0_6c/PASS from prior-run evidence.
prior_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'
spike_0_6c_dir = REPO_DIR / 'data' / 'spike_0_6c'
spike_0_6c_dir.mkdir(parents=True, exist_ok=True)

if prior_history.exists():
    print(f'[recovery] using prior-run history.csv as evidence: {prior_history}')
    !python3 scripts/run_spike_0_6c_1.py \
        --su2-history-csv {prior_history} \
        --result-json {spike_0_6c_dir}/sub_1_result.json \
        --marker-dir {spike_0_6c_dir}
else:
    print(f'[recovery] no prior history.csv at {prior_history}; falling back to fresh probe')
    probe_mesh = spike_0_6c_dir / 'meshes' / 'probe.su2'
    if not probe_mesh.exists():
        !python3 scripts/gen_benchmark_meshes.py --kind probe --out {probe_mesh}
    !python3 scripts/run_spike_0_6c_1.py \
        --probe-mesh {probe_mesh} \
        --probe-workdir {spike_0_6c_dir}/sub_1_run \
        --result-json {spike_0_6c_dir}/sub_1_result.json \
        --marker-dir {spike_0_6c_dir}

# Run the 0.6c aggregator -> writes data/spike_0_6c/PASS iff sub_1 passed.
!python3 scripts/run_spike_0_6c.py \
    --sub-1-json {spike_0_6c_dir}/sub_1_result.json \
    --out {spike_0_6c_dir}/results.json \
    --marker-dir {spike_0_6c_dir}


## Cell 6 — Sub-spike 0.6d.1 (symmetry + dimensional sanity) — ADVISORY (does NOT gate)

Demoted to advisory 2026-05-15 (Note 3). Run it for the Phase-5 record, but its PASS/FAIL does **not** block Phase 4 — the gate is Cell 7 (0.6d.2). Reuses the existing Drive history.csv (no SU2).

In [ ]:
# Re-use the existing 0.6c Cell 8 history.csv (already on Drive from the 2026-05-14 run).
existing_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'
if not existing_history.exists():
    raise FileNotFoundError(
        f'expected the May-2026 Spike 0.6c.2 history.csv at {existing_history}. '
        'This file is the output of the original (now-deferred) 0.6c.2 NACA 0012 '
        'run and should already be on Drive. If it is genuinely gone, you do NOT '
        'need the full 6-12h benchmark — a short SU2 run (even ~50 outer steps) '
        'with the same Tier-1 numerics produces a usable history.csv for the '
        '0.6d.1 symmetry + dimensional checks. See docs/spike_0_6d_protocol.md '
        'sub-spike 0.6d.1 Inputs.'
    )

result_dir = REPO_DIR / 'data' / 'spike_0_6d'
result_dir.mkdir(parents=True, exist_ok=True)

# NACA 0012 effective mass + r_cm — replace with values measured from the actual
# mesh if these defaults don't match Cell 6's mesh.
naca0012_mass_kg = 0.05
naca0012_r_cm_m = 0.25
omega_rad_per_s = 12.5664  # Tier-1 lock

!python3 scripts/run_spike_0_6d_1.py \
    --history-csv {existing_history} \
    --n-cycles 5 \
    --mass-kg {naca0012_mass_kg} \
    --omega-rad-per-s {omega_rad_per_s} \
    --r-cm-m {naca0012_r_cm_m} \
    --envelope-geometry 'NACA 0012 from Spike 0.6c Cell 6 (chord=1m, m=0.05kg, r_cm=0.25m)' \
    --result-json {result_dir}/sub_1_result.json \
    --marker-dir {result_dir}

## Cell 7 — Sub-spike 0.6d.2: 2D thin-plate added-mass frequency-consistency (GATING)

Redesigned 2026-05-15 (see `docs/phase_logs/phase_0_signoff.md` Note 3). The **sole Phase-4 gate**: run the 2D thin-plate at TWO pitching frequencies (ω₁, ω₂) on the SAME mesh / pivot / θ_max, recover the added-mass coefficient `I_a = a_sin/(ω²·θ_max)` from each, and require them to agree (`|ΔI_a|/mean < 0.25`). `I_a` is frequency-independent by construction, and the comparison is SU2-vs-SU2 so the FREESTREAM_PRESS_EQ_ONE `q_ref` cancels — normalization-invariant + parameter-free. ~2 short SU2 runs.

In [ ]:
thin_plate_dir = DRIVE_DIR / 'spike_0_6d' / 'thin_plate_2d'
thin_plate_dir.mkdir(parents=True, exist_ok=True)
thin_plate_mesh = thin_plate_dir / 'thin_plate_2d.su2'

if not thin_plate_mesh.exists():
    print(f'[0.6d.2] thin-plate mesh missing at {thin_plate_mesh}; generating via gmsh')
    !python3 scripts/gen_benchmark_meshes.py \
        --kind thin_plate_2d \
        --out {thin_plate_mesh} \
        --farfield-radius-chord 50.0 \
        --verbose
    if not thin_plate_mesh.exists():
        raise FileNotFoundError(f'mesh generation failed; expected {thin_plate_mesh}')

CHORD_M = 1.0
PIVOT_OFFSET_NORMALIZED = -0.5  # quarter-chord
THETA_MAX = 0.6981             # Tier-1 lock (same for BOTH runs)

# Two frequencies, same plate/pivot/theta. ω₂ = 2·ω₁ — I_a must be identical.
OMEGA_F1 = 12.5664             # Tier-1 lock (2π·f, f=2 Hz)
OMEGA_F2 = 25.1327             # 2·ω₁
N_CYCLES = 5

result_dir = REPO_DIR / 'data' / 'spike_0_6d'
result_dir.mkdir(parents=True, exist_ok=True)


def _run_su2_at(omega: float, tag: str) -> Path:
    """Render + run the 2D thin-plate cfg at one pitching frequency; return history.csv."""
    period = 2.0 * math.pi / omega
    dt = period / 200.0
    time_iter = N_CYCLES * 200
    cfg_text = render_thin_plate_2d_pitching_cfg(
        mesh_filename=str(thin_plate_mesh),
        marker_plate='PLATE',
        marker_farfield='FARFIELD',
        pitching_omega_y=-abs(omega),  # C11 sign lock
        pitching_ampl_y=THETA_MAX,
        motion_origin_x=0.25 * CHORD_M,
        time_step=dt,
        max_time=N_CYCLES * period,
        time_iter=time_iter,
    )
    run_dir = thin_plate_dir / tag
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = run_dir / 'thin_plate_2d_pitching.cfg'
    cfg_path.write_text(cfg_text)
    print(f'[0.6d.2:{tag}] omega={omega:.4f} dt={dt:.4e} cfg -> {cfg_path}')

    log_path = run_dir / 'su2.log'
    with log_path.open('w') as logf:
        proc = subprocess.Popen(
            [str(SU2_BIN), str(cfg_path)],
            cwd=run_dir, stdout=logf, stderr=subprocess.STDOUT,
        )
        while proc.poll() is None:
            time.sleep(60)
            tail = subprocess.run(
                ['tail', '-1', str(log_path)], capture_output=True, text=True
            ).stdout.strip()
            print(f'[0.6d.2:{tag} heartbeat] {tail[:140]}')
    print(f'[0.6d.2:{tag}] SU2 exit={proc.returncode}')
    hist = run_dir / 'history.csv'
    if not hist.exists():
        cands = list(run_dir.glob('history*.csv'))
        if cands:
            hist = cands[0]
    return hist


hist_f1 = _run_su2_at(OMEGA_F1, 'f1')
hist_f2 = _run_su2_at(OMEGA_F2, 'f2')

if hist_f1.exists() and hist_f2.exists():
    !python3 scripts/run_spike_0_6d_2.py \
        --history-csv-f1 {hist_f1} --omega-f1 {OMEGA_F1} \
        --history-csv-f2 {hist_f2} --omega-f2 {OMEGA_F2} \
        --pitching-amplitude-rad {THETA_MAX} \
        --chord-m {CHORD_M} \
        --pivot-offset-normalized {PIVOT_OFFSET_NORMALIZED} \
        --n-cycles {N_CYCLES} \
        --result-json {result_dir}/sub_2_result.json \
        --marker-dir {result_dir}
else:
    print(f'[0.6d.2] missing history.csv (f1={hist_f1.exists()}, f2={hist_f2.exists()}); SU2 likely did not run')


## Cell 8 — Sub-spike 0.6d.3 (incompressible cross-check, advisory)

Optional. Run only if SU2 is built with the incompressible solver enabled. Duplicates the production Tier-1 cfg with `SOLVER = INC_NAVIER_STOKES`, same mesh, motion. Result is advisory; failure does not block Phase 4.

In [ ]:
# This cell is a SKETCH for the operator — actual incompressible-mode cfg generation
# is left as a deferred sub-task (the renderer can produce one but you may want to
# tune INC_NONDIM and reference values manually). Run only if you have a working
# incompressible-mode cfg AND its history.csv. Skip in V1 to keep the gate moving;
# Phase 5 step 62.5 (3-solver cross-check including OpenFOAM) absorbs this work.

comp_history = DRIVE_DIR / 'spike_0_6c' / 'sub_2_run' / 'history.csv'
incomp_history = DRIVE_DIR / 'spike_0_6d' / 'incompressible' / 'history.csv'

result_dir = REPO_DIR / 'data' / 'spike_0_6d'
if comp_history.exists() and incomp_history.exists():
    !python3 scripts/run_spike_0_6d_3.py \
        --comp-history-csv {comp_history} \
        --incomp-history-csv {incomp_history} \
        --result-json {result_dir}/sub_3_result.json
else:
    print('[0.6d.3] SKIPPED — advisory; sub_3 will be marked "not run" by aggregator')
    print(f'  comp history present:   {comp_history.exists()}')
    print(f'  incomp history present: {incomp_history.exists()}')

## Cell 9 — Aggregator + Phase 4 launch gate check

Runs the 0.6d aggregator; writes `data/spike_0_6d/PASS` iff 0.6d.1 AND 0.6d.2 both passed. Then checks the dual-gate (`launch_phase4.py --check`) for end-to-end readiness.

In [ ]:
result_dir = REPO_DIR / 'data' / 'spike_0_6d'

# Gate = sub_2 (frequency-consistency) ONLY. sub_1/sub_3 advisory + optional.
adv_args = []
if (result_dir / 'sub_1_result.json').exists():
    adv_args += ['--sub-1-json', str(result_dir / 'sub_1_result.json')]
if (result_dir / 'sub_3_result.json').exists():
    adv_args += ['--sub-3-json', str(result_dir / 'sub_3_result.json')]

!python3 scripts/run_spike_0_6d.py \
    --sub-2-json {result_dir}/sub_2_result.json \
    {' '.join(adv_args)} \
    --out {result_dir}/results.json \
    --marker-dir {result_dir}

print()
print('=== DUAL-GATE READINESS CHECK ===')
!python3 scripts/launch_phase4.py --check

result = json.loads((result_dir / 'results.json').read_text())
print('\n=== FINAL ===')
print(f"overall_passed (= sub_06d_2 freq-consistency): {result['result']['overall_passed']}")
print(f"sub_06d_2.passed (GATING): {result['result']['sub_06d_2']['passed']}")
s1 = result['result'].get('sub_06d_1')
s3 = result['result'].get('sub_06d_3')
print(f"sub_06d_1 (advisory): {s1['passed'] if s1 else 'not run'}")
print(f"sub_06d_3 (advisory): {s3['passed'] if s3 else 'not run'}")


## Cell 10 — (Optional) commit + push the PASS marker

Only run this if you want the `data/spike_0_6d/PASS` marker committed back to the repo. Requires a GitHub PAT with `repo` scope stored in Colab Secrets as `GITHUB_PAT`.

In [ ]:
PAT = None
try:
    PAT = colab_userdata.get('GITHUB_PAT')
except Exception as e:
    print(f'[commit] GITHUB_PAT secret not configured ({e}); skipping push')

if PAT:
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'config', 'user.email', GIT_EMAIL], check=True
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'config', 'user.name', GIT_USER], check=True
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'add',
         'data/spike_0_6c/', 'data/spike_0_6d/'],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'commit', '-m',
         'Spike 0.6c.1 + 0.6d: PASS markers from Colab run'],
        check=False,  # no-op if nothing changed
    )
    push_url = f'https://{PAT}@github.com/{GIT_USER}/fan-optimization.git'
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'push', push_url, f'HEAD:{GIT_BRANCH}'],
        check=True,
    )
    print(f'[commit] pushed markers to {GIT_BRANCH}')
else:
    print('[commit] skipped (no PAT) — markers are written locally in the repo only')


## Done

If 0.6d.1 + 0.6d.2 both passed AND 0.6c.1 passed previously, `scripts/launch_phase4.py --check` returns 0 — Phase 4 is unblocked. Move on to Phase 1 / Phase 2 / Phase 3 / Phase 4 as the plan dictates.